# Driver Drowsiness Detection - Google Colab Training

This notebook trains a MobileNetV2-based model to detect drowsiness in drivers using eye state classification.

## Setup Instructions:
1. Upload your dataset to Google Drive
2. Mount Google Drive in this notebook
3. Run all cells to train your model
4. Download the trained model when complete


## 1. Setup and Dependencies


In [ ]:
# Install required packages
!pip install tensorflow==2.15.0
!pip install opencv-python
!pip install mediapipe
!pip install matplotlib
!pip install seaborn


In [ ]:
import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger

# Check GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"GPU memory growth: {tf.config.experimental.get_memory_growth(tf.config.list_physical_devices('GPU')[0]) if tf.config.list_physical_devices('GPU') else 'No GPU'}")


## 2. Mount Google Drive and Setup Paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Update these paths based on where you uploaded your dataset
DRIVE_PATH = '/content/drive/MyDrive/Drowsiness-Detection'  # Adjust this path
DATASET_DIR = os.path.join(DRIVE_PATH, 'Driver Drowsiness Dataset (DDD)')
MODEL_DIR = os.path.join(DRIVE_PATH, 'models')
MODEL_PATH = os.path.join(MODEL_DIR, 'drowsiness_mobilenetv2.keras')
LABELS_PATH = os.path.join(MODEL_DIR, 'class_indices.json')

# Create model directory if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

# Check if dataset exists
if os.path.exists(DATASET_DIR):
    print(f"✅ Dataset found at: {DATASET_DIR}")
    # Count images
    drowsy_count = len([f for f in os.listdir(os.path.join(DATASET_DIR, 'Drowsy')) if f.endswith('.png')])
    non_drowsy_count = len([f for f in os.listdir(os.path.join(DATASET_DIR, 'Non Drowsy')) if f.endswith('.png')])
    print(f"📊 Dataset stats: {drowsy_count} drowsy images, {non_drowsy_count} non-drowsy images")
else:
    print(f"❌ Dataset not found at: {DATASET_DIR}")
    print("Please upload your dataset to Google Drive and update the DRIVE_PATH variable")


## 3. Data Preparation and Augmentation


In [ ]:
# Training configuration
IMG_SIZE = 224
BATCH_SIZE = 64  # Increased for GPU
EPOCHS = 15
VALIDATION_SPLIT = 0.2

print(f"Configuration:")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Validation split: {VALIDATION_SPLIT}")


In [ ]:
def build_data_generators():
    """Create data generators with augmentation"""
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        validation_split=VALIDATION_SPLIT,
        rotation_range=15,
        width_shift_range=0.15,
        height_shift_range=0.15,
        zoom_range=0.2,
        shear_range=0.1,
        brightness_range=(0.7, 1.3),
        horizontal_flip=True,
        fill_mode="nearest",
    )

    common_args = dict(
        directory=DATASET_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="binary",
        shuffle=True,
    )

    train_gen = train_datagen.flow_from_directory(
        subset="training",
        **common_args,
    )
    val_gen = train_datagen.flow_from_directory(
        subset="validation",
        **common_args,
    )

    return train_gen, val_gen

# Create data generators
train_gen, val_gen = build_data_generators()

print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Class indices: {train_gen.class_indices}")

# Save class indices
with open(LABELS_PATH, 'w', encoding='utf-8') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print(f"✅ Class indices saved to: {LABELS_PATH}")


## 4. Model Architecture


In [ ]:
def build_model():
    """Build MobileNetV2-based model with transfer learning"""
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    
    # Freeze most layers for stable training
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    # Add custom classification head
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.2)(x)
    output = Dense(1, activation="sigmoid")(x)
    
    model = Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    
    return model

# Build and display model
model = build_model()
model.summary()

# Count trainable parameters
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
total_params = sum([tf.keras.backend.count_params(w) for w in model.weights])
print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")


## 5. Training Configuration


In [ ]:
# Setup callbacks
callbacks = [
    EarlyStopping(
        monitor="val_accuracy", 
        patience=5, 
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss", 
        factor=0.5, 
        patience=3, 
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_PATH, 
        monitor="val_accuracy", 
        save_best_only=True,
        verbose=1
    ),
    CSVLogger(
        os.path.join(MODEL_DIR, 'training_log.csv'),
        append=False
    )
]

print("✅ Callbacks configured:")
print("- Early stopping (patience=5)")
print("- Learning rate reduction")
print("- Model checkpointing")
print("- CSV logging")


## 6. Start Training


In [ ]:
# Calculate steps
steps_per_epoch = math.ceil(train_gen.samples / BATCH_SIZE)
validation_steps = math.ceil(val_gen.samples / BATCH_SIZE)

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Validation steps: {validation_steps}")
print(f"\n🚀 Starting training...")

# Start training
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1,
)

print("\n✅ Training completed!")


## 7. Model Evaluation and Visualization


In [ ]:
# Final evaluation
print("\n📊 Final Model Evaluation:")
val_loss, val_acc = model.evaluate(val_gen, verbose=1)
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Validation Loss: {val_loss:.4f}")


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', color='blue')
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='red')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Save plots
plt.savefig(os.path.join(MODEL_DIR, 'training_history.png'), dpi=300, bbox_inches='tight')
print(f"✅ Training plots saved to: {os.path.join(MODEL_DIR, 'training_history.png')}")


In [ ]:
# Generate predictions for confusion matrix
print("\n🔍 Generating predictions for detailed evaluation...")
val_gen.reset()
predictions = model.predict(val_gen, steps=validation_steps)
predicted_classes = (predictions > 0.5).astype(int).flatten()

# Get true labels
true_classes = val_gen.classes[:len(predicted_classes)]

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Drowsy', 'Drowsy'], 
            yticklabels=['Non-Drowsy', 'Drowsy'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(true_classes, predicted_classes, 
                          target_names=['Non-Drowsy', 'Drowsy']))

# Save confusion matrix
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
print(f"✅ Confusion matrix saved to: {os.path.join(MODEL_DIR, 'confusion_matrix.png')}")


## 8. Download Trained Model


In [ ]:
# List all saved files
print("📁 Files saved to Google Drive:")
for file in os.listdir(MODEL_DIR):
    file_path = os.path.join(MODEL_DIR, file)
    if os.path.isfile(file_path):
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"  - {file} ({size:.2f} MB)")

print(f"\n✅ Model saved to: {MODEL_PATH}")
print(f"✅ Class indices saved to: {LABELS_PATH}")
print("\n📥 To download files:")
print("1. Go to Google Drive")
print("2. Navigate to your Drowsiness-Detection folder")
print("3. Download the 'models' folder")


## 9. Model Usage Instructions

After training, you can use your model with the following code:

```python
import tensorflow as tf
import json
import cv2
import numpy as np

# Load model and class indices
model = tf.keras.models.load_model('drowsiness_mobilenetv2.keras')
with open('class_indices.json', 'r') as f:
    class_indices = json.load(f)

# Preprocess image
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (224, 224))
    img = img.astype(np.float32) / 255.0
    return np.expand_dims(img, axis=0)

# Make prediction
def predict_drowsiness(image_path):
    processed_img = preprocess_image(image_path)
    prediction = model.predict(processed_img)[0][0]
    
    if prediction > 0.5:
        return "Drowsy", prediction
    else:
        return "Non-Drowsy", 1 - prediction

# Example usage
result, confidence = predict_drowsiness('path_to_image.jpg')
print(f"Prediction: {result} (Confidence: {confidence:.2f})")
```
